<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8"><p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">✅ Session 5 — Functions, Scope &amp; Closures · Solutions</p><p style="margin:0;">Worked, runnable solutions for the 12 <strong>Exercises</strong> and 8 <strong>Code Challenges</strong>. Run top to bottom to verify. Try them in <code>01_functions.ipynb</code> first.</p></div>

### Exercises — Solutions

In [ ]:
# E1 (Easy) — build_url(base, **params)
def build_url(base, **params):
    return base + "?" + "&".join(f"{k}={v}" for k, v in params.items())

print(build_url("api", page=1, size=10))   # api?page=1&size=10

In [ ]:
# E2 (Easy) — fix BOTH bugs: mutable default AND the `or []` false-empty pitfall
def add(item, acc=None):
    if acc is None:          # NOT `acc or []` (would discard a passed empty list)
        acc = []
    acc.append(item)
    return acc

print(add(1), add(2))                  # [1] [2] - no accumulation
passed = []
print(add(9, passed) is passed, passed)  # True [9] - passed empty list IS filled

In [ ]:
# E3 (Medium) — make_counter(start=0, step=1)
def make_counter(start=0, step=1):
    n = start - step
    def inc():
        nonlocal n
        n += step
        return n
    return inc

c = make_counter(10, 5)
print(c(), c(), c())                   # 10 15 20

In [ ]:
# E4 (Medium) — apply_n(f, x, n)
def apply_n(f, x, n):
    for _ in range(n):
        x = f(x)
    return x

print(apply_n(lambda v: v*2, 1, 3))    # 8

In [ ]:
# E5 (Medium) — calc(a, b, /, *, op)
def calc(a, b, /, *, op):
    return op(a, b)

print(calc(3, 4, op=max))              # 4

In [ ]:
# E6 (Medium) — trace(fn): wrapper factory (decorator shape)
def trace(fn):
    def wrapper(*args, **kwargs):
        print(f"call {fn.__name__}{args}")
        return fn(*args, **kwargs)
    return wrapper

add = lambda a, b: a + b
print(trace(add)(2, 3))                # prints "call <lambda>(2, 3)", returns 5

In [ ]:
# E7 (Medium) — running_stats() -> (count, mean)
def running_stats():
    total = n = 0
    def add(x):
        nonlocal total, n
        total += x; n += 1
        return (n, total / n)
    return add

s = running_stats()
print(s(10), s(20))                    # (1, 10.0) (2, 15.0)

In [ ]:
# E8 (Hard) — memoize(fn) keyed by *args
def memoize(fn):
    cache = {}
    def wrapper(*args):
        if args not in cache:          # args tuple is hashable (2B)
            cache[args] = fn(*args)
        return cache[args]
    return wrapper

m = memoize(lambda a, b: a + b)
print(m(1, 2), m(1, 2))                # 3 3 (2nd from cache)

In [ ]:
# E9 (Hard) — compose(*funcs) left-to-right
def compose(*funcs):
    def inner(x):
        for f in funcs:
            x = f(x)
        return x
    return inner

print(compose(lambda x: x+1, lambda x: x*2)(3))   # 8

In [ ]:
# E10 (Hard) — partial(fn, *fixed)
def partial(fn, *fixed):
    def inner(*rest):
        return fn(*fixed, *rest)
    return inner

add3 = lambda a, b, c: a + b + c
print(partial(add3, 1)(2, 3), partial(add3, 1, 2)(3))   # 6 6

In [ ]:
# E11 (Hard) — beat late binding, and explain why
# Naive: [lambda: i for i in range(3)] -> all return 2 (they share one i, read at call time).
funcs = [lambda i=i: i for i in range(3)]   # default arg captures the value NOW
print([f() for f in funcs])            # [0, 1, 2]

In [ ]:
# E12 (Hard) — once(fn): run first time only, cache result forever
def once(fn):
    done = False
    result = None
    def wrapper(*args, **kwargs):
        nonlocal done, result
        if not done:
            result = fn(*args, **kwargs)
            done = True
        return result
    return wrapper

calls = []
def work(x):
    calls.append(x)
    return x * 10
w = once(work)
print(w(5), w(6), calls)               # 50 50 [5] - work ran exactly once

### Code Challenges — Solutions

In [ ]:
# C1 (Easy) — flip(fn)
def flip(fn):
    return lambda a, b: fn(b, a)

print(flip(pow)(2, 3))                 # 9

In [ ]:
# C2 (Easy) — negate(pred)
def negate(pred):
    return lambda *a, **k: not pred(*a, **k)

print(negate(str.isdigit)("a"), negate(str.isdigit)("5"))   # True False

In [ ]:
# C3 (Medium) — count_calls(fn)
def count_calls(fn):
    def wrapper(*a, **k):
        wrapper.calls += 1
        return fn(*a, **k)
    wrapper.calls = 0
    return wrapper

w = count_calls(len); w("ab"); w("cde")
print(w.calls)                         # 2

In [ ]:
# C4 (Medium) — group_by(items, key_fn)
from collections import defaultdict
def group_by(items, key_fn):
    g = defaultdict(list)
    for it in items:
        g[key_fn(it)].append(it)
    return dict(g)

print(group_by([1, 2, 3, 4, 5], lambda x: x % 2))   # {1:[1,3,5], 0:[2,4]}

In [ ]:
# C5 (Medium) — with_retry(fn, times)
def with_retry(fn, times):
    def wrapper(*a, **k):
        last = None
        for _ in range(times):
            try:
                return fn(*a, **k)
            except Exception as e:
                last = e
        raise last
    return wrapper

state = {"n": 0}
def flaky():
    state["n"] += 1
    if state["n"] < 3:
        raise ValueError("boom")
    return "ok"
print(with_retry(flaky, 5)(), "after", state["n"], "tries")   # ok after 3 tries

In [ ]:
# C6 (Medium) — pipe(x, *funcs)
def pipe(x, *funcs):
    for f in funcs:
        x = f(x)
    return x

print(pipe(3, lambda x: x+1, lambda x: x*2))   # 8

In [ ]:
# C7 (Hard) — curry3(fn)
def curry3(fn):
    return lambda a: lambda b: lambda c: fn(a, b, c)

print(curry3(lambda a, b, c: a + b + c)(1)(2)(3))   # 6

In [ ]:
# C8 (Hard) — make_stack() -> (push, pop) sharing one hidden list
def make_stack():
    items = []
    def push(x):
        items.append(x)
    def pop():
        return items.pop()
    return push, pop

push, pop = make_stack()
push(1); push(2)
print(pop(), pop())                    # 2 1